# Surgery Log RAG Pipeline: Groq LLM Generation

This notebook connects to your previous FAISS retrieval pipeline.

It uses the saved files:

```text
vector_store/
├── surgery_logs_faiss.index
├── surgery_logs_metadata.csv
└── surgery_logs_embeddings.npy
```

Pipeline:

```text
User Question
→ Hybrid Retrieval
→ Build RAG Context
→ Build LLM Prompt
→ Gemini LLM Generation
→ Grounded Answer
```


## 1. Install required packages

Run this once in your virtual environment:

```bash
pip install pandas numpy faiss-cpu sentence-transformers python-dotenv langchain-google-genai rank-bm25
```


In [1]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq


/home/corpadm/my_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Groq API key

Create a `.env` file in your project root:

```text
Groq_API_KEY=your_google_api_key_here
```

Do not upload `.env` to GitHub.


In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY,
    temperature=0.2
)

print("Groq LLM loaded successfully")

Groq LLM loaded successfully


## 3. Load FAISS index, metadata, and embeddings

This assumes you already ran the previous embedding + FAISS notebook.


In [14]:
PROJECT_DIR = Path("/home/corpadm/my_project/laparoscopic-surgery-rag-assistant")

VECTOR_STORE_DIR = PROJECT_DIR / "vector_store"

FAISS_INDEX_PATH = VECTOR_STORE_DIR / "surgery_logs_faiss.index"
METADATA_PATH = VECTOR_STORE_DIR / "surgery_logs_metadata.csv"
EMBEDDINGS_PATH = VECTOR_STORE_DIR / "surgery_logs_embeddings.npy"

if not FAISS_INDEX_PATH.exists():
    raise FileNotFoundError(f"FAISS index not found: {FAISS_INDEX_PATH}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_PATH}")

if not EMBEDDINGS_PATH.exists():
    raise FileNotFoundError(f"Embeddings file not found: {EMBEDDINGS_PATH}")

loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))
loaded_metadata = pd.read_csv(METADATA_PATH)
loaded_embeddings = np.load(EMBEDDINGS_PATH)

print("Loaded FAISS vectors:", loaded_index.ntotal)
print("Loaded metadata rows:", len(loaded_metadata))
print("Loaded embeddings shape:", loaded_embeddings.shape)

display(loaded_metadata.head())


Loaded FAISS vectors: 610
Loaded metadata rows: 610
Loaded embeddings shape: (610, 768)


,chunk_id,chunk_type,case_id,source_file,surgery_type,surgeon_name,surgery_date,event_start_time,event_end_time,patient_age,patient_bmi,instrument_names,text,vector_id
0,Adrenalectomy_2025-10-14_16-12-00__case_summary,case_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,"[""Cobra_Grasper"", ""Fenestrated_Bipolar_Forceps...",Case Adrenalectomy_2025-10-14_16-12-00 is a la...,0
1,Adrenalectomy_2025-10-14_16-12-00__patient_timing,patient_timing,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,[],Timing and patient details for case Adrenalect...,1
2,Adrenalectomy_2025-10-14_16-12-00__pedal_activity,pedal_activity,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,[],Pedal activity summary for case Adrenalectomy_...,2
3,Adrenalectomy_2025-10-14_16-12-00__instrument_...,instrument_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,"[""Cobra_Grasper"", ""Fenestrated_Bipolar_Forceps...",Instrument usage summary for case Adrenalectom...,3
4,Adrenalectomy_2025-10-14_16-12-00__instrument_1,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:55:00,2025-10-14 16:55:00,59,20.0,"[""Monopolar_Cautery_Hook""]",Instrument event for case Adrenalectomy_2025-1...,4


## 4. Load the same embedding model used during indexing

In [15]:
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)


Embedding model loaded: all-mpnet-base-v2


## 5. Helper functions

In [16]:
def apply_metadata_filters(df, filters=None):
    if not filters:
        return df.copy()

    filtered = df.copy()

    for col, value in filters.items():
        if value is None or col not in filtered.columns:
            continue

        if isinstance(value, str):
            filtered = filtered[
                filtered[col].fillna("").astype(str).str.contains(value, case=False, na=False)
            ]
        elif isinstance(value, list):
            pattern = "|".join([str(v) for v in value])
            filtered = filtered[
                filtered[col].fillna("").astype(str).str.contains(pattern, case=False, na=False)
            ]
        else:
            filtered = filtered[filtered[col] == value]

    return filtered.reset_index(drop=True)


def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_]+", " ", text)
    return text.split()


def normalize_scores(scores):
    scores = np.array(scores, dtype="float32")

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score - min_score == 0:
        return np.ones_like(scores)

    return (scores - min_score) / (max_score - min_score)


## 6. Full Hybrid Retriever

This uses:

```text
Metadata filtering + FAISS semantic search + BM25 keyword search
```


In [17]:
def hybrid_search(
    query,
    top_k=5,
    filters=None,
    semantic_weight=0.65,
    keyword_weight=0.35
):
    start_total = time.time()

    filtered_metadata = apply_metadata_filters(loaded_metadata, filters)

    if filtered_metadata.empty:
        return pd.DataFrame()

    filtered_vector_ids = filtered_metadata["vector_id"].tolist()
    filtered_embeddings = loaded_embeddings[filtered_vector_ids].astype("float32")

    temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
    temp_index.add(filtered_embeddings)

    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    candidate_k = min(max(top_k * 3, top_k), len(filtered_metadata))

    start_faiss = time.time()
    semantic_scores, semantic_indices = temp_index.search(query_embedding, candidate_k)
    faiss_latency_ms = round((time.time() - start_faiss) * 1000, 2)

    semantic_results = filtered_metadata.iloc[semantic_indices[0]].copy()
    semantic_results["semantic_score"] = semantic_scores[0]

    filtered_texts = filtered_metadata["text"].fillna("").astype(str).tolist()
    filtered_tokens = [tokenize(doc) for doc in filtered_texts]
    temp_bm25 = BM25Okapi(filtered_tokens)

    query_tokens = tokenize(query)
    keyword_scores = temp_bm25.get_scores(query_tokens)

    keyword_results = filtered_metadata.copy()
    keyword_results["keyword_score"] = keyword_scores
    keyword_results = keyword_results.sort_values(
        "keyword_score", ascending=False
    ).head(candidate_k)

    combined = pd.merge(
        semantic_results[["vector_id", "semantic_score"]],
        keyword_results[["vector_id", "keyword_score"]],
        on="vector_id",
        how="outer"
    )

    combined["semantic_score"] = combined["semantic_score"].fillna(0)
    combined["keyword_score"] = combined["keyword_score"].fillna(0)

    combined["semantic_score_norm"] = normalize_scores(combined["semantic_score"])
    combined["keyword_score_norm"] = normalize_scores(combined["keyword_score"])

    combined["hybrid_score"] = (
        semantic_weight * combined["semantic_score_norm"]
        + keyword_weight * combined["keyword_score_norm"]
    )

    final_results = pd.merge(
        combined,
        loaded_metadata,
        on="vector_id",
        how="left"
    )

    final_results = final_results.sort_values(
        "hybrid_score", ascending=False
    ).head(top_k)

    final_results["rank"] = range(1, len(final_results) + 1)
    final_results["faiss_latency_ms"] = faiss_latency_ms
    final_results["total_retrieval_latency_ms"] = round((time.time() - start_total) * 1000, 2)

    return final_results.reset_index(drop=True)


## 7. Build RAG context

In [18]:
def build_context(retrieved_df, max_chunks=5):
    if retrieved_df.empty:
        return ""

    context_parts = []

    for _, row in retrieved_df.head(max_chunks).iterrows():
        source = (
            f"Source: case_id={row.get('case_id')}, "
            f"chunk_id={row.get('chunk_id')}, "
            f"chunk_type={row.get('chunk_type')}, "
            f"surgery_type={row.get('surgery_type')}, "
            f"surgeon={row.get('surgeon_name')}, "
            f"source_file={row.get('source_file')}"
        )

        text = row.get("text", "")
        context_parts.append(f"{source}\n{text}")

    return "\n\n---\n\n".join(context_parts)


## 8. Build LLM prompt

In [19]:
def build_llm_prompt(question, context):
    prompt = f"""
You are a healthcare data assistant for laparoscopic surgery event-log analysis.

Your task:
Answer the user's question using only the retrieved surgery-log context.

Strict rules:
1. Do not invent information.
2. If the answer is not available in the retrieved context, say:
   "Not available in the retrieved surgery logs."
3. Mention case_id and source_file when possible.
4. Keep the answer clear, concise, and useful for surgical data review.
5. Do not provide medical advice, diagnosis, or clinical decisions.
6. If multiple records are retrieved, compare them only using the values present in the context.

Retrieved Surgery-Log Context:
{context}

User Question:
{question}

Final Answer:
"""
    return prompt.strip()


## 9. Main RAG answer function

In [20]:
def answer_question_with_rag(
    question,
    top_k=5,
    filters=None,
    semantic_weight=0.65,
    keyword_weight=0.35,
    show_sources=True
):
    retrieved_df = hybrid_search(
        query=question,
        top_k=top_k,
        filters=filters,
        semantic_weight=semantic_weight,
        keyword_weight=keyword_weight
    )

    if retrieved_df.empty:
        return {
            "answer": "No matching surgery logs found for the given query and filters.",
            "retrieved_chunks": retrieved_df,
            "prompt": None,
            "llm_latency_ms": None
        }

    context = build_context(retrieved_df, max_chunks=top_k)
    prompt = build_llm_prompt(question, context)

    start_llm = time.time()
    response = llm.invoke(prompt)
    llm_latency_ms = round((time.time() - start_llm) * 1000, 2)

    answer = response.content

    if show_sources:
        sources = retrieved_df[[
            "rank",
            "hybrid_score",
            "semantic_score",
            "keyword_score",
            "chunk_type",
            "case_id",
            "source_file"
        ]].copy()

        print("Retrieved sources:")
        display(sources)
        print(f"LLM latency: {llm_latency_ms} ms")

    return {
        "answer": answer,
        "retrieved_chunks": retrieved_df,
        "prompt": prompt,
        "llm_latency_ms": llm_latency_ms
    }


## 10. Example: Question without filters

In [21]:
question = "Which instruments were used in the cholecystectomy surgery logs?"

result = answer_question_with_rag(
    question=question,
    top_k=5,
    filters=None
)

print("\nFinal Answer:\n")
print(result["answer"])


Retrieved sources:


,rank,hybrid_score,semantic_score,keyword_score,chunk_type,case_id,source_file
0,1,0.650000,0.788702,0.0,instrument_summary,Cholecystectomy_2025-10-16_16-59-00,Cholecystectomy_2025-10-16_16-59-00.json
1,2,0.645972,0.783815,0.0,instrument_summary,Cholecystectomy_2025-11-10_10-13-00,Cholecystectomy_2025-11-10_10-13-00.json
2,3,0.644866,0.782473,0.0,instrument_summary,Cholecystectomy_2025-10-14_17-07-00,Cholecystectomy_2025-10-14_17-07-00.json
3,4,0.641078,0.777877,0.0,instrument_summary,Cholecystectomy_2025-11-01_11-24-00,Cholecystectomy_2025-11-01_11-24-00.json
4,5,0.627436,0.761323,0.0,instrument_summary,Cholecystectomy_2025-10-12_14-32-00,Cholecystectomy_2025-10-12_14-32-00.json


LLM latency: 809.29 ms

Final Answer:

Based on the retrieved surgery-log context, the following instruments were used in the cholecystectomy surgery logs:

1. Small_Graptor
2. Mega_Needle_Driver
3. Resano_Forceps
4. Endowrist_Stapler_30_Curved_Tip_Instrument
5. Mega_Suturecut_Needle_Driver
6. Suction Irrigation
7. Monopolar_Curved_Scissors
8. Large_Clip_Applier
9. Prograsp_Forceps
10. Tenaculum_Forceps
11. SureForm_45_Instruments
12. Long_Bipolar_Grasper
13. SynchroSeal
14. Potts_Scissors
15. Long_Tip_Forceps


## 11. Example: Question with metadata filters

In [26]:
question = "Which instruments were used in Dr MERAI's cholecystectomy case?"

filters = {
    "surgery_type": "Cholecystectomy",
    "surgeon_name": "MERAI"
}

result = answer_question_with_rag(
    question=question,
    top_k=5,
    filters=filters
)

print("\nFinal Answer:\n")
print(result["answer"])


Retrieved sources:


,rank,hybrid_score,semantic_score,keyword_score,chunk_type,case_id,source_file
0,1,0.867508,0.639309,6.175176,case_summary,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json
1,2,0.805430,0.608063,5.408838,case_summary,Cholecystectomy_2025-11-03_10-02-00,Cholecystectomy_2025-11-03_10-02-00.json
2,3,0.680467,0.685820,0.719206,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json
3,4,0.672954,0.678324,0.709581,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json
4,5,0.656059,0.666375,0.578069,instrument_summary,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json


LLM latency: 636.01 ms

Final Answer:

Instruments used in Dr. MERAI's cholecystectomy case (case_id=Dr.MERAI_Cholecystectomy_20260521_105445): 

- Fenestrated_Bipolar_Forceps
- Medium_Large_Clip_Applier
- Monopolar_Cautery_Hook


## 12. Example: Pedal activity question

In [23]:
question = "How many clutch pedal and camera pedal events were recorded?"

result = answer_question_with_rag(
    question=question,
    top_k=5,
    filters={"surgery_type": "Cholecystectomy"}
)

print("\nFinal Answer:\n")
print(result["answer"])


Retrieved sources:


,rank,hybrid_score,semantic_score,keyword_score,chunk_type,case_id,source_file
0,1,1.000000,0.634997,10.660444,pedal_activity,Today,Today.json
1,2,0.853229,0.494580,10.567969,pedal_activity,Dr.MERAI_Cholecystectomy_20260521_105445,Dr.MERAI_Cholecystectomy_20260521_105445.json
2,3,0.847056,0.497116,10.300882,pedal_activity,Cholecystectomy_2025-10-16_16-59-00,Cholecystectomy_2025-10-16_16-59-00.json
3,4,0.841262,0.491455,10.300882,pedal_activity,Cholecystectomy_2025-12-21_10-41-00,Cholecystectomy_2025-12-21_10-41-00.json
4,5,0.840587,0.490796,10.300882,pedal_activity,Cholecystectomy_2025-10-12_14-32-00,Cholecystectomy_2025-10-12_14-32-00.json


LLM latency: 315.16 ms

Final Answer:

Not available in the retrieved surgery logs.


## 13. Inspect the exact prompt sent to Gemini

In [24]:
print(result["prompt"])


You are a healthcare data assistant for laparoscopic surgery event-log analysis.

Your task:
Answer the user's question using only the retrieved surgery-log context.

Strict rules:
1. Do not invent information.
2. If the answer is not available in the retrieved context, say:
   "Not available in the retrieved surgery logs."
3. Mention case_id and source_file when possible.
4. Keep the answer clear, concise, and useful for surgical data review.
5. Do not provide medical advice, diagnosis, or clinical decisions.
6. If multiple records are retrieved, compare them only using the values present in the context.

Retrieved Surgery-Log Context:
Source: case_id=Today, chunk_id=Today__pedal_activity, chunk_type=pedal_activity, surgery_type=Cholecystectomy, surgeon=Dr.Merai ai, source_file=Today.json
Pedal activity summary for case Today. The log contains 2 clutch pedal press events and 0 camera pedal press events. These events describe intraoperative workflow activity during the surgery.

---

S

## 14. Save query logs

This saves question, answer, latency, and retrieved source IDs.


In [25]:
LOG_PATH = PROJECT_DIR / "rag_query_logs.csv"

def save_rag_log(question, result):
    retrieved = result["retrieved_chunks"]

    source_case_ids = []
    source_files = []
    chunk_ids = []

    if retrieved is not None and not retrieved.empty:
        source_case_ids = retrieved["case_id"].astype(str).tolist()
        source_files = retrieved["source_file"].astype(str).tolist()
        chunk_ids = retrieved["chunk_id"].astype(str).tolist()

    row = {
        "question": question,
        "answer": result["answer"],
        "llm_latency_ms": result.get("llm_latency_ms"),
        "source_case_ids": str(source_case_ids),
        "source_files": str(source_files),
        "chunk_ids": str(chunk_ids)
    }

    log_df = pd.DataFrame([row])

    if LOG_PATH.exists():
        old = pd.read_csv(LOG_PATH)
        log_df = pd.concat([old, log_df], ignore_index=True)

    log_df.to_csv(LOG_PATH, index=False)
    print("Saved log to:", LOG_PATH)


save_rag_log(question, result)


Saved log to: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/rag_query_logs.csv


# Production notes

For production, split this notebook into:

```text
src/retriever.py
src/rag_prompt.py
src/llm_generation.py
app.py
```

Recommended improvements:

- Add RAGAS evaluation
- Add answer feedback
- Add hallucination checks
- Add citations in UI
- Keep real patient/case data out of GitHub
